In [1]:
import json
import re
import codecs
import urllib.request
import requests 
from bs4 import BeautifulSoup # for HTML parsing
import time # for sleep
from tqdm import tqdm # for progress tracker
import urllib3 # to disable SSL warnings 

# Disable the SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
from ons_discovery import (DiscoveryClient, is_png, ons_url, prepare_run,
                           validate_config, write_json_files)

with open('config.json', 'r', encoding='utf-8') as config_file:
    config = validate_config(json.load(config_file))

In [ ]:
import re

MONTH = (
    r"(?:January|February|March|April|May|June|July|August|"
    r"September|October|November|December)"
)
YEAR = r"(?:19|20)\d{2}"
DATE_POINT = rf"(?:(?:[0-3]?\d\s+)?{MONTH}\s+)?{YEAR}"
DATE_PERIOD = (
    rf"(?:{MONTH}\s+to\s+{MONTH}\s+{YEAR}|"
    rf"{DATE_POINT}(?:\s+to\s+{DATE_POINT})?)"
)
DATE_SUFFIX = re.compile(
    rf"\s*[:,]\s*(?:(?:between|(?:financial\s+)?year\s+ending)\s+)?"
    rf"{DATE_PERIOD}(?:\s+and\s+{DATE_PERIOD})*\s*$|"
    rf"\s*[:,]\s*mid-{YEAR}\s*$",
    re.IGNORECASE,
)


def derive_series(title):
    cleaned_title = title.strip()
    series = DATE_SUFFIX.sub("", cleaned_title).strip()
    return series or cleaned_title

In [ ]:
def extract_edition(html, title):
    heading = BeautifulSoup(html, "lxml").find("h1")
    if heading is None:
        return None

    heading_text = " ".join(heading.get_text(" ", strip=True).split())
    title_prefix = " ".join(title.split()).rstrip(":") + ":"
    if heading_text.startswith(title_prefix):
        return heading_text[len(title_prefix):].strip() or None

    suffix = DATE_SUFFIX.search(heading_text)
    if suffix is not None:
        return suffix.group().lstrip(" :,").strip() or None
    return None

In [ ]:
from datetime import datetime


def publication_date(article):
    value = article.get('release_date')
    message = f"Invalid release_date for {article.get('doc_uri', 'unknown release')}: {value!r}"
    if not isinstance(value, str):
        raise ValueError(message)
    try:
        return datetime.fromisoformat(value.replace('Z', '+00:00')).date()
    except ValueError as error:
        raise ValueError(message) from error


def mark_latest(records):
    newest_dates = {}
    dated_records = []
    for article in records:
        released = publication_date(article)
        series = article['series']
        if series not in newest_dates or released > newest_dates[series]:
            newest_dates[series] = released
        dated_records.append((article, released))

    for article, released in dated_records:
        article['latest'] = released == newest_dates[article['series']]

In [ ]:
from urllib.parse import urlsplit


def extract_themes(doc_uri):
    segments = [segment for segment in urlsplit(doc_uri).path.split('/') if segment]
    boundary = next((index for index, segment in enumerate(segments)
                     if segment in ('articles', 'bulletins', 'methodologies')), None)
    if boundary is None or not 1 <= boundary <= 3:
        raise ValueError(f"Unsupported theme hierarchy in doc_uri: {doc_uri}")

    hierarchy = segments[:boundary]
    return {name: hierarchy[index] if index < len(hierarchy) else None
            for index, name in enumerate(('theme', 'topic', 'subtopic'))}

In [ ]:
try:
    with open('articles-and-dvcs.json', 'r', encoding='utf-8') as input_file:
        seed = json.load(input_file)
except FileNotFoundError:
    seed = []

try:
    with open('discovery-state.json', 'r', encoding='utf-8') as state_file:
        state = json.load(state_file)
except FileNotFoundError:
    state = None

client = DiscoveryClient()
try:
    cached_publications, results, next_state = prepare_run(seed, config, state, client)
except Exception:
    client.session.close()
    raise

In [ ]:
existing_data = []
if config['mode'] == 'update':
    try:
        with open('data.json', 'r', encoding='utf-8') as existing_file:
            existing_data = json.load(existing_file)
    except FileNotFoundError:
        existing_data = []

pages_lookup = {}
for article in sorted(existing_data, key=publication_date):
    doc_uri = ons_url(article['doc_uri'])
    if doc_uri is None:
        raise ValueError(f"Invalid existing doc_uri: {article['doc_uri']}")
    article['doc_uri'] = doc_uri
    pages_lookup[doc_uri] = article
pages = []

In [ ]:
try:
    for article in tqdm(results, desc="Processing articles"):
        if not article['vis_urls']:
            pages_lookup.pop(article['doc_uri'], None)
            continue

        html = client.get_text(article['doc_uri'])
        article['edition'] = extract_edition(html, article['title'])
        article['vis'] = []
        for vis in tqdm(article['vis_urls'], desc="Fetching visualizations", leave=False):
            if 'datadownload.xlsx' in vis:
                continue
            url = ons_url(vis)
            html = client.get_text(url)
            soup = BeautifulSoup(html, "lxml")
            meta = soup.find("meta", attrs={"name": "template"})
            template = meta.get("content") if meta else None
            article['vis'].append({'url': vis, 'chart_type': template})

            if template is None:
                meta = soup.find("meta", attrs={"name": lambda value: value and "template" in value.lower()}) \
                    or soup.find("meta", attrs={"property": lambda value: value and "template" in value.lower()})
                template = meta.get("content") if meta else None

        pages.append(article)
except Exception:
    client.session.close()
    raise

Processing articles:  13%|█▎        | 223/1674 [02:06<05:43,  4.23it/s]  C:\Users\rickas\AppData\Local\Temp\ipykernel_24064\4030569374.py:19: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
Processing articles: 100%|██████████| 1674/1674 [12:08<00:00,  2.30it/s]


In [ ]:
try:
    for article in pages:
        pages_lookup[article['doc_uri']] = article

    merged = sorted(pages_lookup.values(), key=publication_date, reverse=True)

    for article in merged:
        themes = extract_themes(article['doc_uri'])
        for field in themes:
            article.pop(field, None)
        article.update(themes)
        article['series'] = derive_series(article['title'])
        article.setdefault('edition', None)
        article['vis'] = [vis for vis in article.get('vis', []) if not is_png(vis['url'])]
        article.pop('vis_urls', None)

    mark_latest(merged)

    write_json_files([
        ('data.json', merged),
        ('articles-and-dvcs.json', cached_publications),
        ('discovery-state.json', next_state),
    ])
finally:
    client.session.close()